# Bloque 1 — Estructuras de Datos, Expresiones Regulares y Funciones Lambda
---
## Proyecto: Red Social Ficticia

Trabajamos con los datos de una **red social ficticia** generada para practicar análisis de datos con Python. Los datos están repartidos en tres ficheros dentro de la carpeta `data/`:

- **`usuarios.csv`** — un registro por usuario, con campos como 
    - `id_usuario` — identificador único del usuario (numérico como string, p. ej. '1'). Se usa para relacionar con publicaciones.csv y conexiones.json.
    - `nombre` — nombre de pila (p. ej. 'Ana').
    - `apellidos` — uno o dos apellidos (p. ej. 'Fernandez Gonzalez').
    - `edad` — edad del usuario en años, como string (p. ej. '24'') → son str(), conviértela con int()
    - `ciudad` — ciudad de residencia; puede venir vacía ('') si el usuario no la indicó.
    - `pais` — país de residencia (en este dataset, siempre 'Espana').
    - `genero` — código de género, 'F' o 'M'.
    - `fecha_registro` — fecha en que el usuario se dio de alta, formato 'YYYY-MM-DD'.
    - `tipo_cuenta` — tipo de cuenta: 'free', 'premium', 'creator', etc.
    - `seguidores` — número de seguidores, como string (p. ej. '2016') → son str(),conviértelo con int()
    - `seguidos` — número de cuentas a las que sigue, también como string.
    - `publicaciones_totales` — número total de publicaciones hechas por el usuario, como string.
    - `activo` — indica si la cuenta está activa, como string 'True'/'False' (compáralo con == 'True').
    - `bio` — texto libre de biografía/descripción del perfil.


- **`publicaciones.csv`** — un registro por publicación, con 

    - `id_publicacion` — identificador único de la publicación (numérico como string).
    - `id_usuario` — id del usuario que hizo la publicación; es la clave para relacionar cada publicación con su autor en usuarios.csv.
    - `fecha` — fecha de publicación, formato 'YYYY-MM-DD' (se puede ordenar directamente como string).
    - `tipo_contenido` — tipo de contenido publicado: 'foto', 'video', 'texto', etc.
    - `likes` — número de "me gusta", como string con formato decimal (p. ej. '1329.0') → conviértelo con float() (no int() directamente, porque tiene el .0).
    - `comentarios` — número de comentarios, también como string decimal (p. ej. '354.0') → igual, usa float().
    - `compartidos` — número de veces compartida, como string entero (p. ej. '105') → puedes usar int().
    - `visualizaciones` — número de visualizaciones, como string entero.
    - `hashtags` — cadena con los hashtags de la publicación separados por comas (p. ej. '#datascience,#cloud'); para trabajar con ellos individualmente hay que hacer .split(','). Puede venir vacía si la publicación no tiene hashtags.

- **`conexiones.json`** — relaciones de seguimiento entre usuarios: una lista de objetos       
    - `id_usuario` — id del usuario (aquí sí es un número entero real, no un string, porque viene de JSON, no de CSV).
    - `siguiendo` — lista de ids (enteros) de los usuarios a los que este usuario sigue. Es decir, representa las relaciones de "seguir" en la red social, una por cada usuario. `{"id_usuario": ..., "siguiendo": [ids de usuarios a los que sigue]}`.

- **Punto clave:**
    
    - Todos los campos numéricos y booleanos llegan como **texto** al leerlos con el módulo `csv` (p. ej. `'2016'` o `'True'`), así que a menudo habrá que convertirlos (`int(...)`) o compararlos como string antes de operar con ellos.
    - Fíjate que `likes` y `comentarios` vienen con decimales `(.0)` mientras que `compartidos` y `visualizaciones` no — así que al convertir usa `float()` para los dos primeros e `int()` para los otros dos (o `int(float(valor)`) si quieres unificar).
    - En el caso de fichero conexiones:
        - Al venir de JSON (no de CSV), los números ya son int, no strings — no hace falta convertirlos con int().
        - No es un diccionario único, sino una lista de diccionarios (uno por usuario), así que para buscar las conexiones de un usuario concreto hay que recorrer la lista y comparar c['id_usuario'] con el id buscado (como se hace en la función siguiendo_de del ejercicio 3.3).
        - Es la estructura que permite calcular cosas como seguidores en común, recomendaciones, o cuántos usuarios siguen a alguien en concreto.

---
### Objetivos
En este bloque trabajamos con los datos de la red social usando **solo la librería estándar de Python** (sin Pandas).

Contenido:
- **Listas**: operaciones, comprensiones, slicing
- **Diccionarios**: índices, agrupaciones, transformaciones
- **Conjuntos**: operaciones de conjunto sobre datos reales
- **Tuplas y namedtuples**: registros inmutables
- **Funciones lambda** con `filter`, `map`, `sorted` y `reduce`
- **Expresiones regulares**: extracción y búsqueda de hashtags con `re`

---
### Bloque de código inicial
Esta celda carga en memoria los tres ficheros de datos de la red social ficticia (usuarios, publicaciones y conexiones) usando solo la librería estándar de Python, sin necesidad de Pandas. 
Busca automáticamente la carpeta `data/` sea cual sea el directorio desde el que se ejecute el notebook, y avisa con un error claro si no encuentra los ficheros. Al final deja listas las variables `usuarios`, `publicaciones` y `conexiones` que se usarán en todos los ejercicios del bloque

In [1]:
from pathlib import Path
import csv
import json
import re
from functools import reduce
from collections import namedtuple

# Carga de datos con el modulo csv estandar (SIN pandas)
posibles_carpetas_datos = [
    Path("../data"),
    Path("data"),
    Path("Proy_analisis_Red_Social_Ficticia/data"),
]

DATA_DIR = next(
    (carpeta for carpeta in posibles_carpetas_datos if (carpeta / "usuarios.csv").exists()),
    None,
)

if DATA_DIR is None:
    rutas_probadas = "\n".join(str(carpeta / "usuarios.csv") for carpeta in posibles_carpetas_datos)
    raise FileNotFoundError(
        "No se encontro usuarios.csv. Rutas probadas:\n"
        f"{rutas_probadas}\n\n"
        f"Directorio actual del kernel: {Path.cwd()}"
    )

def cargar_csv(ruta):
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

usuarios = cargar_csv(DATA_DIR / "usuarios.csv")
publicaciones = cargar_csv(DATA_DIR / "publicaciones.csv")

with open(DATA_DIR / "conexiones.json", encoding="utf-8") as f:
    conexiones = json.load(f)

print(f"Carpeta de datos:       {DATA_DIR}")
print(f"Usuarios cargados:      {len(usuarios)}")
print(f"Publicaciones cargadas: {len(publicaciones)}")
print(f"Conexiones cargadas:    {len(conexiones)}")
print(f"\nEjemplo de usuario:")
for k, v in list(usuarios[0].items())[:6]:
    print(f"  {k}: {v}")


Carpeta de datos:       ../data
Usuarios cargados:      100
Publicaciones cargadas: 300
Conexiones cargadas:    100

Ejemplo de usuario:
  id_usuario: 1
  nombre: Ana
  apellidos: Fernandez Gonzalez
  edad: 24
  ciudad: Granada
  pais: Espana


---
## Ejercicios básicos
### Sección 1 — Listas

Las listas son la estructura secuencial mutable más usada en Python.

In [7]:
# ─── Ejercicio 1.1 ─────────────────────────────────────────────────────────
# Practica el bucle 'while' con un índice manual (patrón habitual cuando se
# leen datos con input() hasta que el usuario decide terminar).
# Recorre 'usuarios' con un bucle while y un índice 'i' que empiece en 0.
# Añade el nombre de cada usuario a la lista 'nombres_while' con .append(),
# incrementando 'i' en cada vuelta, hasta procesar los 10 primeros usuarios.

i=0
nombres_while = []
while(i<=10):
    nombres_while.append(usuarios[i].get("nombre"))
    i += 1
print(f"Nombres de los 10 primeros usuarios:       {nombres_while}")



Nombres de los 10 primeros usuarios:       ['Ana', 'Pablo', 'Miguel', 'Pablo', 'Alejandro', 'Elena', 'Raul', 'Daniel', 'Patricia', 'Ivan', 'Cristina']


In [8]:
# ─── Ejercicio 1.2 ─────────────────────────────────────────────────────────
# Practica el uso de índices explícitos con range(len(...)).
# Crea una lista de tuplas (posicion, nombre) para los 5 primeros usuarios,
# donde 'posicion' empieza en 0.
# Una vez terminado haz el mismo ejerciio utilizando enumerate() en vez de range(len(...)).

posiciones_nombres = None 
tuplasUsuarios = []
for posiciones_nombres in range(5):
    tuplasUsuarios.append((posiciones_nombres, usuarios[posiciones_nombres].get("nombre")))

print(f"Nombres de los 5 primeros usuarios:       {tuplasUsuarios}")

posiciones_nombres = None 
posiciones_nombres_lista = usuarios[:5]
tuplasUsuarios = []
for posiciones_nombres, posiciones_nombres_lista in enumerate(posiciones_nombres_lista):
    tuplasUsuarios.append((posiciones_nombres, usuarios[posiciones_nombres].get("nombre")))

print(f"Nombres de los 5 primeros usuarios:       {tuplasUsuarios}")


Nombres de los 5 primeros usuarios:       [(0, 'Ana'), (1, 'Pablo'), (2, 'Miguel'), (3, 'Pablo'), (4, 'Alejandro')]
Nombres de los 5 primeros usuarios:       [(0, 'Ana'), (1, 'Pablo'), (2, 'Miguel'), (3, 'Pablo'), (4, 'Alejandro')]


In [5]:
# ─── Ejercicio 1.3 ─────────────────────────────────────────────────────────
# Practica el patrón de acumulador manual con variable "flag": calcula el
# usuario con MENOS seguidores SIN usar la función min().
# Recorre 'usuarios' con un for y usa una variable booleana 'primero' para
# inicializar el acumulador 'usuario_menos_seguidores' la primera vuelta.

usuario_menos_seguidores = None
primero = True
for usuario in usuarios:
    if not primero and int(usuario.get('seguidores')) < int(usuario_menos_seguidores.get('seguidores')):
        usuario_menos_seguidores = usuario

    if primero:
            usuario_menos_seguidores = usuario
            primero = False

print(f"{usuario_menos_seguidores}")

{'id_usuario': '52', 'nombre': 'Miguel', 'apellidos': 'Hernandez Alvarez', 'edad': '48', 'ciudad': 'Sevilla', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-12-10', 'tipo_cuenta': 'free', 'seguidores': '31', 'seguidos': '900', 'publicaciones_totales': '232', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}


In [9]:
# ─── Ejercicio 1.4 ─────────────────────────────────────────────────────────
# Usa SLICING para obtener:
# a) los 5 primeros usuarios
# b) los 10 últimos usuarios
# c) uno de cada tres (cada tercer usuario)

primeros_5  = usuarios[:5]
ultimos_10  = usuarios[-10:]  
cada_tres   = usuarios[::3]  

print(f"Primeros 5: {primeros_5}")
print(f"Ultimos 10: {ultimos_10}")
print(f"Cada 3: {cada_tres}")



Primeros 5: [{'id_usuario': '1', 'nombre': 'Ana', 'apellidos': 'Fernandez Gonzalez', 'edad': '24', 'ciudad': 'Granada', 'pais': 'Espana', 'genero': 'F', 'fecha_registro': '2022-05-14', 'tipo_cuenta': 'free', 'seguidores': '2016', 'seguidos': '52', 'publicaciones_totales': '15', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '2', 'nombre': 'Pablo', 'apellidos': 'Fernandez Alvarez', 'edad': '55', 'ciudad': '', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-11-30', 'tipo_cuenta': 'premium', 'seguidores': '3446', 'seguidos': '452', 'publicaciones_totales': '174', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '3', 'nombre': 'Miguel', 'apellidos': 'Ruiz Ruiz', 'edad': '34', 'ciudad': 'Alicante', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2022-02-14', 'tipo_cuenta': 'premium', 'seguidores': '802', 'seguidos': '100', 'publicaciones_totales': '282', 'activo': 'True', 'bio': 'Estudiante de Big Data'}, {'id_usuario': '4', 'no

In [12]:
# ─── Ejercicio 1.5 ─────────────────────────────────────────────────────────
# El campo 'activo' viene como string 'True' / 'False' desde el CSV.
# Crea una lista 'activos' con solo los usuarios cuyo campo 'activo' == 'True'.

activos = []
for usuario in usuarios:
    if usuario.get('activo') == 'True':
        activos.append(usuario)

print(f"Activos {activos}")


Activos [{'id_usuario': '1', 'nombre': 'Ana', 'apellidos': 'Fernandez Gonzalez', 'edad': '24', 'ciudad': 'Granada', 'pais': 'Espana', 'genero': 'F', 'fecha_registro': '2022-05-14', 'tipo_cuenta': 'free', 'seguidores': '2016', 'seguidos': '52', 'publicaciones_totales': '15', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '2', 'nombre': 'Pablo', 'apellidos': 'Fernandez Alvarez', 'edad': '55', 'ciudad': '', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-11-30', 'tipo_cuenta': 'premium', 'seguidores': '3446', 'seguidos': '452', 'publicaciones_totales': '174', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '3', 'nombre': 'Miguel', 'apellidos': 'Ruiz Ruiz', 'edad': '34', 'ciudad': 'Alicante', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2022-02-14', 'tipo_cuenta': 'premium', 'seguidores': '802', 'seguidos': '100', 'publicaciones_totales': '282', 'activo': 'True', 'bio': 'Estudiante de Big Data'}, {'id_usuario': '4', 'nombre

In [14]:
# ─── Ejercicio 1.6 ─────────────────────────────────────────────────────────
# Crea una lista 'populares' con los usuarios que tienen MÁS de 1000 seguidores.
# Nota: el campo 'seguidores' viene como string desde el CSV → convierte a int.

populares = []
for usuario in usuarios:
    if int(usuario.get('seguidores')) > 1000:
        populares.append(usuario)

print(f"Populares {populares}")



Populares [{'id_usuario': '1', 'nombre': 'Ana', 'apellidos': 'Fernandez Gonzalez', 'edad': '24', 'ciudad': 'Granada', 'pais': 'Espana', 'genero': 'F', 'fecha_registro': '2022-05-14', 'tipo_cuenta': 'free', 'seguidores': '2016', 'seguidos': '52', 'publicaciones_totales': '15', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '2', 'nombre': 'Pablo', 'apellidos': 'Fernandez Alvarez', 'edad': '55', 'ciudad': '', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-11-30', 'tipo_cuenta': 'premium', 'seguidores': '3446', 'seguidos': '452', 'publicaciones_totales': '174', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '5', 'nombre': 'Alejandro', 'apellidos': 'Vazquez Perez', 'edad': '52', 'ciudad': 'Bilbao', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-08-03', 'tipo_cuenta': 'free', 'seguidores': '5000', 'seguidos': '724', 'publicaciones_totales': '285', 'activo': 'True', 'bio': 'Amante del codigo limpio'}, {'id_usuario': '6', 

In [17]:
# ─── Ejercicio 1.7 ─────────────────────────────────────────────────────────
# Crea una lista de tuplas (nombre, seguidores) para todos los usuarios.
# Usa comprensión de lista.

pares_nombre_seg = None
tuplas_usuarios = [(usuario.get('nombre'), usuario.get('seguidores')) for usuario in usuarios]
print(f"Usuarios: {tuplas_usuarios}")


Usuarios: [('Ana', '2016'), ('Pablo', '3446'), ('Miguel', '802'), ('Pablo', '385'), ('Alejandro', '5000'), ('Elena', '2594'), ('Raul', '2162'), ('Daniel', '5163'), ('Patricia', '2414'), ('Ivan', '14035'), ('Cristina', '484'), ('Ivan', '28730'), ('Carlos', '37301'), ('Adrian', '4427'), ('Maria', '506'), ('Laura', '4518'), ('Carlos', '2182'), ('Raul', '2579'), ('Elena', '992'), ('Elena', '3252'), ('Alejandro', '2869'), ('Carmen', '2176'), ('Luis', '3479'), ('Antonio', '305'), ('Raul', '1276'), ('Carmen', '327'), ('Alejandro', '4183'), ('Fernando', '4976'), ('Marta', '3543'), ('Antonio', '3454'), ('Marcos', '1760'), ('Pablo', '210'), ('Carlos', '3492'), ('Beatriz', '48138'), ('Elena', '3613'), ('Daniel', '4455'), ('Sergio', '3204'), ('Lucia', '1055'), ('Fernando', '3734'), ('Luis', '2035'), ('Carmen', '2558'), ('Carmen', '4928'), ('Sergio', '1263'), ('Laura', '3702'), ('Alejandro', '46952'), ('Ana', '2420'), ('Isabel', '2069'), ('Claudia', '19546'), ('Sara', '850'), ('Luis', '1232'), ('Cr

In [22]:
# ─── Ejercicio 1.8 ─────────────────────────────────────────────────────────
# El campo 'hashtags' de cada publicación es una cadena '#ia,#python,...'
# Crea una lista PLANA con todos los hashtags de todas las publicaciones.
# Incluye los duplicados (se eliminarán en la sección de conjuntos).

todos_hashtags = []  
for publicacion in publicaciones:
    if publicacion.get('hashtags'):
        for hashtag in publicacion.get('hashtags').split(','):
            todos_hashtags.append(hashtag)

print(f"Hashtags: {todos_hashtags}")


Hashtags: ['#datascience', '#cloud', '#innovacion', '#cloud', '#datascience', '#innovacion', '#analisis', '#bigdata', '#python', '#deeplearning', '#datos', '#deeplearning', '#ia', '#startup', '#developer', '#python', '#analisis', '#bigdata', '#deeplearning', '#ia', '#programacion', '#datos', '#tecnologia', '#innovacion', '#bigdata', '#cloud', '#datos', '#ia', '#bigdata', '#developer', '#startup', '#tecnologia', '#machinelearning', '#developer', '#innovacion', '#python', '#datos', '#tecnologia', '#coding', '#programacion', '#machinelearning', '#python', '#datascience', '#startup', '#machinelearning', '#ia', '#developer', '#tecnologia', '#startup', '#machinelearning', '#cloud', '#developer', '#cloud', '#innovacion', '#datascience', '#tecnologia', '#programacion', '#startup', '#datascience', '#datos', '#ia', '#startup', '#ia', '#bigdata', '#coding', '#python', '#ia', '#innovacion', '#ia', '#datascience', '#bigdata', '#datos', '#innovacion', '#bigdata', '#startup', '#cloud', '#machinelearn

---
## Sección 2 — Diccionarios

Los diccionarios permiten acceso O(1) por clave y son perfectos para crear índices.

In [25]:
# ─── Ejercicio 2.1 ─────────────────────────────────────────────────────────
# Crea un diccionario 'indice_usuarios' que mapee id_usuario → nombre.
# Así podemos buscar el nombre de un usuario por su id en O(1).

indice_usuarios = {usuario.get('id_usuario'): usuario.get('nombre') for usuario in usuarios}  

print(f"Indice de usuarios: {indice_usuarios}")


Indice de usuarios: {'1': 'Ana', '2': 'Pablo', '3': 'Miguel', '4': 'Pablo', '5': 'Alejandro', '6': 'Elena', '7': 'Raul', '8': 'Daniel', '9': 'Patricia', '10': 'Ivan', '11': 'Cristina', '12': 'Ivan', '13': 'Carlos', '14': 'Adrian', '15': 'Maria', '16': 'Laura', '17': 'Carlos', '18': 'Raul', '19': 'Elena', '20': 'Elena', '21': 'Alejandro', '22': 'Carmen', '23': 'Luis', '24': 'Antonio', '25': 'Raul', '26': 'Carmen', '27': 'Alejandro', '28': 'Fernando', '29': 'Marta', '30': 'Antonio', '31': 'Marcos', '32': 'Pablo', '33': 'Carlos', '34': 'Beatriz', '35': 'Elena', '36': 'Daniel', '37': 'Sergio', '38': 'Lucia', '39': 'Fernando', '40': 'Luis', '41': 'Carmen', '42': 'Carmen', '43': 'Sergio', '44': 'Laura', '45': 'Alejandro', '46': 'Ana', '47': 'Isabel', '48': 'Claudia', '49': 'Sara', '50': 'Luis', '51': 'Cristina', '52': 'Miguel', '53': 'Andrea', '54': 'Cristina', '55': 'Ana', '56': 'Alejandro', '57': 'Nuria', '58': 'Ivan', '59': 'Elena', '60': 'Claudia', '61': 'Alicia', '62': 'Alba', '63': 'Sa

In [28]:
# ─── Ejercicio 2.2 ─────────────────────────────────────────────────────────
# Crea un diccionario 'por_ciudad' que mapee ciudad → [lista de nombres].
# Ignora los usuarios sin ciudad (campo vacío o None).

por_ciudad = {}  
for usuario in usuarios:
    ciudad = usuario.get('ciudad')
    if usuario.get('ciudad'):
        if ciudad not in por_ciudad:
            por_ciudad[ciudad] = []
        por_ciudad[ciudad].append(usuario.get('nombre'))

print(f"Indice de ciudades: {por_ciudad}")

Indice de ciudades: {'Granada': ['Ana', 'Raul', 'Elena', 'Sergio', 'Andrea', 'Ana', 'Andrea', 'Manuel', 'Miguel'], 'Alicante': ['Miguel', 'Patricia', 'Carlos', 'Carlos', 'Isabel', 'Cristina', 'Nuria', 'Pablo'], 'Vitoria': ['Pablo', 'Fernando', 'Alicia', 'Sara', 'Cristina', 'Alba', 'Laura'], 'Bilbao': ['Alejandro', 'Luis', 'Antonio', 'Beatriz', 'Claudia', 'Claudia'], 'Las Palmas': ['Elena', 'Elena', 'Lucia', 'Carmen', 'Ana', 'Cristina', 'Carmen', 'Fernando', 'Ivan', 'Roberto'], 'Malaga': ['Raul', 'Ivan', 'Daniel', 'Carmen', 'Patricia'], 'Palma': ['Daniel', 'Carlos', 'Ivan', 'Jorge', 'Luis', 'Laura', 'Alicia'], 'Cordoba': ['Ivan', 'Pablo', 'Ana', 'Manuel'], 'Murcia': ['Cristina', 'Alejandro', 'Carmen', 'Luis', 'Sara', 'Alejandro'], 'Sevilla': ['Adrian', 'Fernando', 'Alejandro', 'Miguel', 'Elena', 'Alba', 'Antonio'], 'Barcelona': ['Maria', 'Laura', 'Luis', 'Nuria', 'Sara', 'Miguel', 'Claudia', 'Marcos', 'Fernando'], 'Valencia': ['Carmen', 'Antonio', 'Marcos', 'Sergio', 'Cristina', 'Antoni

In [31]:
# ─── Ejercicio 2.3 ─────────────────────────────────────────────────────────
# Crea una función que reciba un id_usuario (como string) y devuelva
# el diccionario completo del usuario usando el índice creado en 2.1.

indice_usuarios = {usuario.get('id_usuario'): usuario for usuario in usuarios}
def buscar_usuario(id_str):
    """Devuelve el dict del usuario con ese id, o None si no existe."""
    if id_str in indice_usuarios:
        return indice_usuarios[id_str]
    return None

print(buscar_usuario('1'))      
print(buscar_usuario('9999')) 


{'id_usuario': '1', 'nombre': 'Ana', 'apellidos': 'Fernandez Gonzalez', 'edad': '24', 'ciudad': 'Granada', 'pais': 'Espana', 'genero': 'F', 'fecha_registro': '2022-05-14', 'tipo_cuenta': 'free', 'seguidores': '2016', 'seguidos': '52', 'publicaciones_totales': '15', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}
None


In [32]:
# ─── Ejercicio 2.4 ─────────────────────────────────────────────────────────
# Añade un campo calculado 'ratio_engagement' a cada usuario.
# ratio_engagement = seguidores / (seguidos + 1)  (el +1 evita división por cero)
# Trabaja sobre una COPIA de la lista para no modificar el original.

usuarios_con_ratio = []
for u in usuarios:
    u_copia = dict(u)  # copia del diccionario
    u_copia['ratio_engagement'] = float(u.get('seguidores')) / (float(u.get('seguidos')) +1)
    usuarios_con_ratio.append(u_copia)

print(usuarios_con_ratio) 

[{'id_usuario': '1', 'nombre': 'Ana', 'apellidos': 'Fernandez Gonzalez', 'edad': '24', 'ciudad': 'Granada', 'pais': 'Espana', 'genero': 'F', 'fecha_registro': '2022-05-14', 'tipo_cuenta': 'free', 'seguidores': '2016', 'seguidos': '52', 'publicaciones_totales': '15', 'activo': 'True', 'bio': 'Explorando el mundo de la IA', 'ratio_engagement': 38.0377358490566}, {'id_usuario': '2', 'nombre': 'Pablo', 'apellidos': 'Fernandez Alvarez', 'edad': '55', 'ciudad': '', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-11-30', 'tipo_cuenta': 'premium', 'seguidores': '3446', 'seguidos': '452', 'publicaciones_totales': '174', 'activo': 'True', 'bio': 'Explorando el mundo de la IA', 'ratio_engagement': 7.607064017660044}, {'id_usuario': '3', 'nombre': 'Miguel', 'apellidos': 'Ruiz Ruiz', 'edad': '34', 'ciudad': 'Alicante', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2022-02-14', 'tipo_cuenta': 'premium', 'seguidores': '802', 'seguidos': '100', 'publicaciones_totales': '282', 'activo': 

---
## Sección 3 — Conjuntos

Los conjuntos (`set`) garantizan unicidad y soportan operaciones de teoría de conjuntos.

In [35]:
# ─── Ejercicio 3.1 ─────────────────────────────────────────────────────────
# Crea un set 'ciudades_unicas' con todas las ciudades distintas de los usuarios.
# Ignora los valores vacíos.

ciudades_unicas = set(usuario.get('ciudad') for usuario in usuarios if usuario.get('ciudad'))  

print(ciudades_unicas) 

{'Palma', 'Las Palmas', 'Malaga', 'Cordoba', 'Barcelona', 'Murcia', 'Madrid', 'Valladolid', 'Granada', 'Valencia', 'Bilbao', 'Zaragoza', 'Sevilla', 'Vitoria', 'Alicante'}


In [39]:
# ─── Ejercicio 3.2 ─────────────────────────────────────────────────────────
# Crea un set 'hashtags_unicos' a partir de la lista 'todos_hashtags' del ejercicio 1.8.
# Muestra cuántos hashtags únicos hay vs el total con duplicados.

todos_hashtags = []  
for publicacion in publicaciones:
    if publicacion.get('hashtags'):
        for hashtag in publicacion.get('hashtags').split(','):
            todos_hashtags.append(hashtag)

print(f"Hashtags lista: {todos_hashtags}")

hashtags_unicos_set = set() 

for publicacion in publicaciones:
    if publicacion.get('hashtags'):
        for hashtag in publicacion.get('hashtags').split(','):
            hashtags_unicos_set.add(hashtag)

print(f"Hashtags set: {hashtags_unicos_set}")


Hashtags lista: ['#datascience', '#cloud', '#innovacion', '#cloud', '#datascience', '#innovacion', '#analisis', '#bigdata', '#python', '#deeplearning', '#datos', '#deeplearning', '#ia', '#startup', '#developer', '#python', '#analisis', '#bigdata', '#deeplearning', '#ia', '#programacion', '#datos', '#tecnologia', '#innovacion', '#bigdata', '#cloud', '#datos', '#ia', '#bigdata', '#developer', '#startup', '#tecnologia', '#machinelearning', '#developer', '#innovacion', '#python', '#datos', '#tecnologia', '#coding', '#programacion', '#machinelearning', '#python', '#datascience', '#startup', '#machinelearning', '#ia', '#developer', '#tecnologia', '#startup', '#machinelearning', '#cloud', '#developer', '#cloud', '#innovacion', '#datascience', '#tecnologia', '#programacion', '#startup', '#datascience', '#datos', '#ia', '#startup', '#ia', '#bigdata', '#coding', '#python', '#ia', '#innovacion', '#ia', '#datascience', '#bigdata', '#datos', '#innovacion', '#bigdata', '#startup', '#cloud', '#machin

In [42]:
# ─── Ejercicio 3.3 ─────────────────────────────────────────────────────────
# Usa el fichero de conexiones para encontrar los usuarios seguidos en COMÚN
# por el usuario con id=1 y el usuario con id=2.
# conexiones es una lista de {id_usuario: int, siguiendo: [int, ...]}

def siguiendo_de(id_usuario):
    """Devuelve el set de ids que sigue el usuario dado."""
    for c in conexiones:
        if c.get('id_usuario') == int(id_usuario):
            return set(c.get('siguiendo', []))
    return set()

lista_usuarios = usuarios[:2]
lista_id1= siguiendo_de(lista_usuarios[0].get('id_usuario'))
lista_id2= siguiendo_de(lista_usuarios[1].get('id_usuario'))
comunes = lista_id1 & lista_id2
no_comunes  = lista_id1 ^ lista_id2

print(f"Comunes: {comunes}")
print(f"No comunes: {no_comunes}")

Comunes: {90, 47}
No comunes: {4, 9, 13, 15, 18, 21, 22, 26, 33, 42, 45, 46, 49, 50, 58, 62, 64, 65, 70, 73, 78, 81, 82, 84, 85, 89, 94, 95, 96, 98, 100}


In [2]:
# ─── Ejercicio 3.4 ─────────────────────────────────────────────────────────
# Encuentra los usuarios que son seguidos por AL MENOS 3 usuarios distintos.
# Itera sobre todas las conexiones y cuenta cuántas veces aparece cada id.

populares_seguidos = None  
cont = {}

for c in conexiones:
    for id_seguido in set(c.get('siguiendo', [])):
        cont[id_seguido] = cont.get(id_seguido, 0) + 1

populares_seguidos = {id_user for id_user, total in cont.items() if total >= 3}

print(f"Usuarios seguidos por al menos 3: {populares_seguidos}")
print(f"Total: {len(populares_seguidos)}")

Usuarios seguidos por al menos 3: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100}
Total: 100


---
## Sección 4 — Tuplas y namedtuples

Las tuplas son inmutables. Las `namedtuple` añaden nombres a los campos, haciendo el código más legible.

In [8]:
# ─── Ejercicio 4.1 ─────────────────────────────────────────────────────────
# Define un namedtuple 'RegistroUsuario' con los campos:
#   id_usuario, nombre, ciudad, seguidores, tipo_cuenta
# Crea una lista de RegistroUsuario a partir de 'usuarios'.

from collections import namedtuple

RegistroUsuario = namedtuple(
    "RegistroUsuario",
    ["id_usuario", "nombre", "ciudad", "seguidores", "tipo_cuenta"]
)

registros = [
    RegistroUsuario(
        id_usuario=int(u["id_usuario"]),
        nombre=u["nombre"],
        ciudad=u.get("ciudad", ""),
        seguidores=int(u["seguidores"]),
        tipo_cuenta=u["tipo_cuenta"]
    )
    for u in usuarios
]

print(registros[0])



RegistroUsuario(id_usuario=1, nombre='Ana', ciudad='Granada', seguidores=2016, tipo_cuenta='free')


In [11]:
# ─── Ejercicio 4.2 ─────────────────────────────────────────────────────────
# Demuestra la INMUTABILIDAD de las tuplas:
# Intenta modificar el campo 'seguidores' del primer registro.
# Captura la excepción con try/except y muestra el mensaje de error.

primer_registro = registros[0]

try:
    # Intentamos reasignar el valor (esto provocará un AttributeError)
    primer_registro.seguidores = 50000
except AttributeError as e:
    print(f"Error capturado (inmutabilidad demostrada): {e}")

Error capturado (inmutabilidad demostrada): can't set attribute


In [10]:
# ─── Ejercicio 4.3 ─────────────────────────────────────────────────────────
# Usa DESEMPAQUETADO de tupla para extraer los campos del primer registro.
# Asigna cada campo a una variable distinta.

id_u, nombre_u, ciudad_u, seg_u, tipo_u = registros[0]

print(f"ID:          {id_u}")
print(f"Nombre:      {nombre_u}")
print(f"Ciudad:      {ciudad_u}")
print(f"Seguidores:  {seg_u}")
print(f"Tipo cuenta: {tipo_u}")

ID:          1
Nombre:      Ana
Ciudad:      Granada
Seguidores:  2016
Tipo cuenta: free


---
## Sección 5 — Funciones Lambda

`lambda`, `filter`, `map`, `sorted` y `reduce` permiten procesar listas de forma concisa y funcional.

In [5]:
# ─── Ejercicio 5.1 ─────────────────────────────────────────────────────────
# Ordena 'usuarios' por número de seguidores de MAYOR a MENOR usando sorted() + lambda.

usuarios_por_seg = sorted(usuarios, key=lambda u: int(u['seguidores']), reverse=True)

print(f"Usuarios ordenados por seguidores: {usuarios_por_seg}")

Usuarios ordenados por seguidores: [{'id_usuario': '34', 'nombre': 'Beatriz', 'apellidos': 'Navarro Hernandez', 'edad': '53', 'ciudad': 'Bilbao', 'pais': 'Espana', 'genero': 'F', 'fecha_registro': '2022-07-10', 'tipo_cuenta': 'creator', 'seguidores': '48138', 'seguidos': '285', 'publicaciones_totales': '384', 'activo': 'True', 'bio': 'Estudiante de IABD'}, {'id_usuario': '45', 'nombre': 'Alejandro', 'apellidos': 'Gomez Torres', 'edad': '18', 'ciudad': 'Sevilla', 'pais': 'Argentina', 'genero': 'M', 'fecha_registro': '2022-04-12', 'tipo_cuenta': 'creator', 'seguidores': '46952', 'seguidos': '520', 'publicaciones_totales': '284', 'activo': 'True', 'bio': 'Analista de datos'}, {'id_usuario': '87', 'nombre': 'Fernando', 'apellidos': 'Lopez Martinez', 'edad': '27', 'ciudad': 'Las Palmas', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2021-07-11', 'tipo_cuenta': 'creator', 'seguidores': '40074', 'seguidos': '473', 'publicaciones_totales': '338', 'activo': 'True', 'bio': 'Estudiante de B

In [6]:
# ─── Ejercicio 5.2 ─────────────────────────────────────────────────────────
# Usa filter() + lambda para obtener solo los usuarios de tipo 'creator'.

creators = filter(lambda u: u.get('tipo_cuenta') == 'creator', usuarios) 

print(f"Creators: {list(creators)}")

Creators: [{'id_usuario': '8', 'nombre': 'Daniel', 'apellidos': 'Diaz Diaz', 'edad': '47', 'ciudad': 'Palma', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-10-25', 'tipo_cuenta': 'creator', 'seguidores': '5163', 'seguidos': '758', 'publicaciones_totales': '58', 'activo': 'True', 'bio': 'Freelance developer'}, {'id_usuario': '10', 'nombre': 'Ivan', 'apellidos': 'Gonzalez Ruiz', 'edad': '28', 'ciudad': 'Cordoba', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-05-11', 'tipo_cuenta': 'creator', 'seguidores': '14035', 'seguidos': '351', 'publicaciones_totales': '250', 'activo': 'True', 'bio': 'Amante del codigo limpio'}, {'id_usuario': '12', 'nombre': 'Ivan', 'apellidos': 'Rodriguez Torres', 'edad': '30', 'ciudad': 'Malaga', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2022-02-03', 'tipo_cuenta': 'creator', 'seguidores': '28730', 'seguidos': '468', 'publicaciones_totales': '460', 'activo': 'True', 'bio': 'Desarrolladora de software | Python lover'}, {'id_usuario'

In [7]:
# ─── Ejercicio 5.3 ─────────────────────────────────────────────────────────
# Usa map() + lambda para crear una lista de etiquetas:
#   Si seguidores >= 5000 → 'influencer', si >= 1000 → 'popular', si no → 'normal'

def etiqueta(u):
    seguidores = int(u['seguidores'])
    if seguidores >= 5000:
        return 'influencer'
    elif seguidores >= 1000:
        return 'popular'
    else:
        return 'normal'

etiquetas = list(map(etiqueta, usuarios))

print(etiquetas) 



['popular', 'popular', 'normal', 'normal', 'influencer', 'popular', 'popular', 'influencer', 'popular', 'influencer', 'normal', 'influencer', 'influencer', 'popular', 'normal', 'popular', 'popular', 'popular', 'normal', 'popular', 'popular', 'popular', 'popular', 'normal', 'popular', 'normal', 'popular', 'popular', 'popular', 'popular', 'popular', 'normal', 'popular', 'influencer', 'popular', 'popular', 'popular', 'popular', 'popular', 'popular', 'popular', 'popular', 'popular', 'popular', 'influencer', 'popular', 'popular', 'influencer', 'normal', 'popular', 'influencer', 'normal', 'popular', 'popular', 'popular', 'popular', 'normal', 'popular', 'popular', 'popular', 'popular', 'popular', 'normal', 'popular', 'popular', 'popular', 'popular', 'popular', 'normal', 'normal', 'popular', 'popular', 'popular', 'popular', 'normal', 'normal', 'influencer', 'popular', 'normal', 'popular', 'popular', 'popular', 'normal', 'popular', 'popular', 'popular', 'influencer', 'popular', 'popular', 'norm

In [8]:
# ─── Ejercicio 5.4 ─────────────────────────────────────────────────────────
# Usa reduce() para calcular el TOTAL de seguidores de todos los usuarios.

total_seguidores = reduce(lambda total, u: total + int(u['seguidores']), usuarios, 0)

print(f"Total de seguidores: {total_seguidores}")


Total de seguidores: 518887


In [9]:
# ─── Ejercicio 5.5 ─────────────────────────────────────────────────────────
# Ordena las publicaciones por fecha de publicación (de más antigua a más reciente).
# El campo 'fecha' es una cadena 'YYYY-MM-DD' → se puede ordenar como string.

pubs_ordenadas = sorted(publicaciones, key=lambda p: p['fecha'])

print(f"Publicaciones ordenadas por fecha: {pubs_ordenadas}")

Publicaciones ordenadas por fecha: [{'id_publicacion': '52', 'id_usuario': '32', 'fecha': '2023-01-07', 'tipo_contenido': 'video', 'likes': '1513.0', 'comentarios': '244.0', 'compartidos': '98', 'visualizaciones': '7893', 'hashtags': '#machinelearning,#programacion,#cloud'}, {'id_publicacion': '250', 'id_usuario': '62', 'fecha': '2023-01-07', 'tipo_contenido': 'historia', 'likes': '3324.0', 'comentarios': '444.0', 'compartidos': '35', 'visualizaciones': '30100', 'hashtags': '#ia,#bigdata'}, {'id_publicacion': '232', 'id_usuario': '22', 'fecha': '2023-01-08', 'tipo_contenido': 'reel', 'likes': '1577.0', 'comentarios': '42.0', 'compartidos': '16', 'visualizaciones': '8328', 'hashtags': '#developer,#innovacion'}, {'id_publicacion': '34', 'id_usuario': '37', 'fecha': '2023-01-09', 'tipo_contenido': 'foto', 'likes': '4894.0', 'comentarios': '334.0', 'compartidos': '110', 'visualizaciones': '19811', 'hashtags': '#coding,#deeplearning,#machinelearning'}, {'id_publicacion': '181', 'id_usuario'

---
## Sección 6 — Expresiones regulares

El módulo estándar `re` permite localizar patrones de texto. En una red social resulta útil para extraer etiquetas y buscar publicaciones concretas.

Patrones que usaremos:

```python
r'#[\w]+'             # hashtag: # seguido de letras, números o _
re.findall(patron, texto)
re.search(patron, texto, flags=re.IGNORECASE)
```

Usa cadenas *raw* (`r'...'`) para que las barras invertidas del patrón se interpreten correctamente.

In [10]:
# ─── Ejercicio R.1 ─────────────────────────────────────────────────────────
# Extrae todos los hashtags de este texto usando re.findall().
# El patrón debe encontrar # seguido de letras, números o _. Normaliza el
# resultado a minúsculas y conserva el orden, incluidos los duplicados.

import re

texto = 'Aprende #Python, #IA y #datos_2026. También #python.'

patron = r'#\w+'
hashtags_regex = [h.lower() for h in re.findall(patron, texto)]

print(hashtags_regex) 



['#python', '#ia', '#datos_2026', '#python']


In [11]:
# ─── Ejercicio R.2 ─────────────────────────────────────────────────────────
# Filtra las publicaciones que contienen exactamente el hashtag buscado.
# La búsqueda debe ignorar mayúsculas, pero '#ia' NO debe encontrar '#iabd'.
# Pista: re.escape(hashtag) convierte el hashtag en un patrón seguro y
# (?!\w) evita que después continúen letras, números o _.

import re

hashtag_buscado = '#ia'

patron = r'' + re.escape(hashtag_buscado) + r'(?!\w)'

pubs_con_ia = [
    p for p in publicaciones 
    if p.get('hashtags') and re.search(patron, p.get('hashtags'), flags=re.IGNORECASE)
]

print(f"Publicaciones encontradas: {len(pubs_con_ia)}")



Publicaciones encontradas: 65


---
## Entrega — `../scripts/pipeline.py` (Bloque 1)

Implementa las siguientes funciones en `../scripts/pipeline.py` y ejecuta:
```bash
python -m pytest ../tests/test_bloque1.py -v
```

| Función | Descripción |
|---|---|
| `usuarios_activos(lista)` | Filtra usuarios con `activo == 'True'` |
| `agrupar_por_ciudad(lista)` | Devuelve `{ciudad: [nombres]}` |
| `hashtags_unicos(lista_pub)` | Devuelve `set` de hashtags únicos |
| `top_por_seguidores(lista, n)` | Devuelve los n usuarios con más seguidores |
| `extraer_hashtags_regex(texto)` | Extrae y normaliza hashtags con `re.findall` |
| `publicaciones_con_hashtag(lista_pub, hashtag)` | Filtra por hashtag completo sin distinguir mayúsculas |

## Ejercicios avanzados

Estos ejercicios combinan varias estructuras y funciones. Resuélvelos después de completar los ejercicios básicos.

### A.1 Informe de actividad por ciudad
Crea un diccionario anidado con esta estructura:

```python
{
    'Madrid': {
        'usuarios': 12,
        'activos': 10,
        'seguidores_totales': 25000,
        'media_seguidores': 2083.33
    }
}
```

Agrupa los usuarios por ciudad, ignora ciudades vacías y calcula los cuatro valores usando listas, diccionarios y `lambda`.

In [6]:
usuarios_por_ciudad = {}

for usuario in usuarios:
    if usuario.get('ciudad'):
        ciudad = usuario.get('ciudad')
        if ciudad not in usuarios_por_ciudad:
            usuarios_por_ciudad[ciudad] = []
        usuarios_por_ciudad[ciudad].append(usuario)

diccionario_por_ciudad = {}

for ciudad, lista_usuarios in usuarios_por_ciudad.items():

    num_usuarios = len(lista_usuarios)
    activos = lambda lista: sum(1 for u in lista if u.get("activo", 'True') == 'True')
    seg_tot = lambda lista: sum(int(u.get("seguidores", 0)) for u in lista)
    media_seguidores = seg_tot(lista_usuarios) / num_usuarios if num_usuarios > 0 else 0

    diccionario_por_ciudad[ciudad] = {
        'usuarios': num_usuarios,
        'activos': activos(lista_usuarios),
        'seguidores_totales': seg_tot(lista_usuarios),
        'media_seguidores': media_seguidores
    }

print(f"Diccionario por ciudad: {diccionario_por_ciudad}")

Diccionario por ciudad: {'Granada': {'usuarios': 9, 'activos': 6, 'seguidores_totales': 27030, 'media_seguidores': 3003.3333333333335}, 'Alicante': {'usuarios': 8, 'activos': 7, 'seguidores_totales': 76050, 'media_seguidores': 9506.25}, 'Vitoria': {'usuarios': 7, 'activos': 7, 'seguidores_totales': 15064, 'media_seguidores': 2152.0}, 'Bilbao': {'usuarios': 6, 'activos': 6, 'seguidores_totales': 79881, 'media_seguidores': 13313.5}, 'Las Palmas': {'usuarios': 10, 'activos': 7, 'seguidores_totales': 60231, 'media_seguidores': 6023.1}, 'Malaga': {'usuarios': 5, 'activos': 5, 'seguidores_totales': 45178, 'media_seguidores': 9035.6}, 'Palma': {'usuarios': 7, 'activos': 7, 'seguidores_totales': 22993, 'media_seguidores': 3284.714285714286}, 'Cordoba': {'usuarios': 4, 'activos': 4, 'seguidores_totales': 27148, 'media_seguidores': 6787.0}, 'Murcia': {'usuarios': 6, 'activos': 5, 'seguidores_totales': 11370, 'media_seguidores': 1895.0}, 'Sevilla': {'usuarios': 7, 'activos': 6, 'seguidores_totale

### A.2 Recomendaciones por conexiones
Para cada usuario, encuentra hasta cinco recomendaciones: personas que sigue alguno de sus contactos pero que todavía no sigue él. No incluyas al propio usuario y ordena las recomendaciones por número de seguidores.


In [2]:
mapa_siguiendo = {item["id_usuario"]: set(item["siguiendo"]) for item in conexiones}
mapa_usuarios = {int(u["id_usuario"]): u for u in usuarios}

recomendaciones_por_usuario = {}

for usuario in usuarios:
    id_actual = int(usuario["id_usuario"])
    siguiendo_actual = mapa_siguiendo.get(id_actual, set())
    
    candidatos = set()
    for contacto in siguiendo_actual:
        candidatos.update(mapa_siguiendo.get(contacto, set()))
    
    candidatos_validos = candidatos - siguiendo_actual - {id_actual}
    
    top_5_ids = sorted(
        candidatos_validos,
        key=lambda id_c: int(mapa_usuarios[id_c].get("seguidores", 0)) if id_c in mapa_usuarios else 0,
        reverse=True
    )[:5]
    
    recomendaciones_por_usuario[usuario["nombre"]] = [
        mapa_usuarios[id_c]["nombre"] for id_c in top_5_ids if id_c in mapa_usuarios
    ]

for nombre, recs in list(recomendaciones_por_usuario.items())[:5]:
    print(f"{nombre}: {recs}")

Ana: ['Beatriz', 'Alejandro', 'Fernando', 'Carlos', 'Cristina']
Pablo: ['Beatriz', 'Alejandro', 'Carlos', 'Ivan', 'Claudia']
Miguel: ['Beatriz', 'Fernando', 'Carlos', 'Ivan', 'Claudia']
Alejandro: ['Beatriz', 'Alejandro', 'Fernando', 'Carlos', 'Cristina']
Elena: ['Beatriz', 'Fernando', 'Carlos', 'Cristina', 'Ivan']


### A.3 Pipeline funcional de publicaciones
Construye una función que reciba la lista de publicaciones y devuelva las cinco etiquetas más utilizadas. Debe encadenar `map`, `filter`, `sorted` y un diccionario de conteos. Compara el resultado con una solución basada en conjuntos.


In [ ]:
def top_5_hashtags_funcional(pubs):
    cadenas_hashtags = map(lambda p: p.get("hashtags", ""), pubs)
    
    cadenas_validas = filter(lambda s: bool(s and s.strip()), cadenas_hashtags)
    
    lista_etiquetas = []
    for cadena in cadenas_validas:
        lista_etiquetas.extend([tag.strip() for tag in cadena.split(",") if tag.strip()])
    
    conteo = {}
    for tag in lista_etiquetas:
        conteo[tag] = conteo.get(tag, 0) + 1
        
    top_5 = sorted(conteo.items(), key=lambda item: item[1], reverse=True)[:5]
    return top_5


resultado_funcional = top_5_hashtags_funcional(publicaciones)

print("Top 5 Pipeline funcional:", resultado_funcional)


Top 5 Pipeline funcional: [('#datascience', 73), ('#tecnologia', 69), ('#startup', 68), ('#programacion', 66), ('#bigdata', 65)]


### A.4 Limpieza de texto
Crea una función `normalizar_hashtag(texto)` que elimine espacios, convierta a minúsculas y garantice que el resultado comienza por `#`. Aplica la función con `map` y elimina los resultados vacíos o repetidos.

In [4]:
def normalizar_hashtag(texto):
    if not texto:
        return ""
    limpio = texto.replace(" ", "").strip().lower()
    
    if not limpio or limpio == "#":
        return ""
    
    if not limpio.startswith("#"):
        limpio = f"#{limpio}"
        
    return limpio

ejemplos = ["  DataScience ", "python", "#IA", "   ", "#MachineLearning", "python", "#", "  #CLOUD  "]

hashtags_normalizados = list(map(normalizar_hashtag, ejemplos))

hashtags_finales = list(dict.fromkeys(filter(bool, hashtags_normalizados)))

print("Originales: ", ejemplos)
print("Normalizados sin vacíos ni repetidos:", hashtags_finales)

Originales:  ['  DataScience ', 'python', '#IA', '   ', '#MachineLearning', 'python', '#', '  #CLOUD  ']
Normalizados sin vacíos ni repetidos: ['#datascience', '#python', '#ia', '#machinelearning', '#cloud']


### A.5 Validación
Comprueba que todos los ids de las publicaciones pertenecen a usuarios existentes y devuelve dos listas: publicaciones válidas y publicaciones con errores. No modifiques las listas originales.

In [5]:
ids_usuarios_existentes = {str(u["id_usuario"]) for u in usuarios}

publicaciones_validas = []
publicaciones_con_errores = []

for pub in publicaciones:
    id_autor = str(pub.get("id_usuario", ""))
    
    if id_autor in ids_usuarios_existentes:
        publicaciones_validas.append(pub)
    else:
        publicaciones_con_errores.append(pub)

print(f"Total publicaciones analizadas:  {len(publicaciones)}")
print(f"Publicaciones válidas:           {len(publicaciones_validas)}")
print(f"Publicaciones con errores:       {len(publicaciones_con_errores)}")

if publicaciones_con_errores:
    print("\nEjemplo de publicación con error:", publicaciones_con_errores[0])

Total publicaciones analizadas:  300
Publicaciones válidas:           300
Publicaciones con errores:       0
